# CIFAR Image Generator on Google Colab

This notebook allows you to train and test the **CIFAR Image Generator** on Google Colab's infrastructure (GPU recommended).

### Check GPU Availability

In [1]:
!nvidia-smi

Fri Mar 27 02:52:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   38C    P8             16W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Setting Up the Workspace

In [2]:
# 2. Mount Drive and Setup Repository
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define paths
drive_path = "/content/drive/MyDrive/" # You can change this path
user = "Bhavikupadhyay"
repo = "cifar-image-generator"
repo_full_path = os.path.join(drive_path, repo)

# Create the base directory if it doesn't exist
if not os.path.exists(drive_path):
    os.makedirs(drive_path)

# Navigate to Drive path
%cd {drive_path}

if not os.path.exists(repo):
    print(f"Cloning {repo} into Drive for the first time...")
    # NOTE: If the repo is PRIVATE, you must use a token:
    # !git clone https://YOUR_TOKEN_HERE@github.com/{user}/{repo}.git
    !git clone https://github.com/{user}/{repo}.git
else:
    print(f"Repository already exists at {repo_full_path}. Pulling latest changes...")
    %cd {repo}
    !git pull

# Ensure we are inside the repo directory
%cd {repo_full_path}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive
Repository already exists at /content/drive/MyDrive/cifar-image-generator. Pulling latest changes...
/content/drive/MyDrive/cifar-image-generator
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 8 (delta 7), reused 8 (delta 7), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 3.11 KiB | 1024 bytes/s, done.
From https://github.com/Bhavikupadhyay/cifar-image-generator
   63b9661..1a0532b  main       -> origin/main
Updating 63b9661..1a0532b
Fast-forward
 inference.py                 |   7 +-
 src/calculate_fid.py         |  83 ++++++------
 src/config.py                |  14 ++-
 train.py                     |  90 +++++++++----
 train_and_sample_colab.ipynb | 294 +------------------------------------------
 5 files changed, 130 insertions(

### Install Dependencies

In [3]:
# For Colab, only need to install wandb. 
%pip install wandb
%pip install torchmetrics[image]

### Run the training loop

In [ ]:
from src import Config
from train import train
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

cfg = Config(
    use_wandb=True,
    use_amp=True,          # FP16 — now effective with GroupNorm replacing BatchNorm
    use_compile=True,      # torch.compile — fuses ops, reduces kernel launch overhead
    use_ema=True,
    fid_every_epochs=25,
    fid_num_samples=128,
)

print("Starting training on Colab...")
print("="*30, "\n")
train(cfg)

### Inference

In [ ]:
timestamp = '2026-02-16_07-27-22' # obtain from the above cell

In [ ]:
!python inference.py --run_name {timestamp} --num_samples 64 --device cuda

### Visualize the Results

In [ ]:
import os
from IPython.display import Image, display

img_path = f'runs/{timestamp}/inference/inference_sample.png'
if os.path.exists(img_path):
    display(Image(filename=img_path))
else:
    print("Image not found. Check if inference ran successfully.")

In [ ]:
# Calculate FID Score for the current run
!PYTHONPATH=. python src/calculate_fid.py {timestamp} --num_samples 1000